# Heat Stress and Cattle Slaughter Analysis

**Purpose**: Analyze the relationship between heat stress indicators and cattle slaughter in USDA regions 4 (Southeast) and 6 (South Central).

**Data Source**: Uses analysis-ready data from `01b_experimental_design_from_processed.ipynb`

**⚠️ Important Note on Data Format**:
- This notebook automatically converts timedelta64 values (stored as nanoseconds) to hours
- **Recommended**: Regenerate your CSV files using the updated `01b_experimental_design_from_processed.ipynb` 
  which now loads data with proper hour conversion via `src/analysis.py`
- Legacy CSV files with nanosecond values will be auto-converted during loading

**Analysis Approach**:
1. Exploratory Data Analysis (EDA)
2. Time series visualization
3. Correlation analysis
4. Panel regression models (requires statsmodels/linearmodels)
5. Effect size estimation
6. Regional comparisons

**Key Variables**:
- **Outcome**: `log_slaughter_beef_dairy` (log-transformed weekly cattle slaughter)
- **Predictors** (all in hours per week after conversion):
  - Daytime heat: `day_above_30c_hrs_sum` (0-168 hrs), `day_above_35c_hrs_sum` (0-168 hrs)
  - Nighttime recovery: `night_bad_hrs_sum` (0-70 hrs), `night_poor_recovery_hrs_sum` (0-70 hrs)
  - VPD: `vpd_afternoon_mean_mean` (kPa)
  - TAS: `tas` (thermal acclimation state, 0-1)
  - Interactions: `day_heat_x_vpd`, `bad_night_into_heat_sum`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import sys
from scipy import stats

# Try importing statsmodels (may have scipy version conflict)
try:
    from statsmodels.regression.linear_model import OLS
    from statsmodels.tools import add_constant
    import statsmodels.api as sm
    HAS_STATSMODELS = True
except ImportError as e:
    HAS_STATSMODELS = False
    print("⚠️  statsmodels import error (likely scipy version conflict):")
    print(f"   {e}")
    print("\n   To fix, update your packages:")
    print("   conda update statsmodels scipy  # if using conda")
    print("   pip install --upgrade statsmodels scipy  # if using pip")
    print("\n   Regression analysis will be skipped.\n")

# Try to import PanelOLS from linearmodels (modern approach)
HAS_PANEL_OLS = False
if HAS_STATSMODELS:
    try:
        from linearmodels.panel import PanelOLS
        HAS_PANEL_OLS = True
    except ImportError:
        HAS_PANEL_OLS = False
        print("⚠️  linearmodels not installed. Panel regression will use OLS with fixed effects.")
        print("   To install: pip install linearmodels\n")

warnings.filterwarnings('ignore')

# Add parent directory to path for config import
sys.path.append('../..')
import config

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 3)

print("✓ Libraries loaded")
if not HAS_STATSMODELS:
    print("⚠️  Regression analysis disabled (statsmodels not available)")

## 1. Load Analysis-Ready Data

Load the merged climate-cattle dataset produced by the experimental design notebook.

In [ ]:
# Load analysis-ready data
# NOTE: Update filename to match your processed data
# This example uses summer 2020, but you can process the full date range
data_file = config.PROCESSED_WEEKLY_DIR / 'analysis_ready_2020_summer.csv'

if data_file.exists():
    df = pd.read_csv(data_file)
    df['week_ending'] = pd.to_datetime(df['week_ending'])
    
    # Fix timedelta64 columns that may be stored as large integers (nanoseconds)
    # These columns should represent hours, not nanoseconds
    hour_columns = [
        'day_above_30c_hrs_sum', 'day_above_35c_hrs_sum',
        'night_bad_hrs_sum', 'night_poor_recovery_hrs_sum'
    ]
    
    # Check if values are unreasonably large (indicating nanoseconds)
    for col in hour_columns:
        if col in df.columns:
            # First, ensure the column is numeric
            df[col] = pd.to_numeric(df[col], errors='coerce')
            
            # If mean value > 1000, it's likely in nanoseconds (should be 0-168 for hours/week)
            if df[col].mean() > 1000:
                print(f"⚠️  Converting {col} from nanoseconds to hours...")
                # Convert nanoseconds to hours: divide by 3.6e12 (ns -> hours)
                df[col] = df[col] / 3.6e12
            
            # Ensure final dtype is float64
            df[col] = df[col].astype(np.float64)
    
    # Ensure other numeric columns are properly typed
    numeric_columns = ['log_slaughter_beef_dairy', 'slaughter_beef_dairy', 
                       'vpd_afternoon_mean_mean', 'tas']
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    print(f"Data loaded: {data_file.name}")
    print(f"  Shape: {df.shape}")
    print(f"  Date range: {df['week_ending'].min()} to {df['week_ending'].max()}")
    print(f"  Regions: {df['region'].unique()}")
    print(f"  Total weeks: {len(df)}")
    
    # Check for missing values in key variables
    key_vars = ['log_slaughter_beef_dairy', 'tas', 'day_above_30c_hrs_sum', 
                'night_bad_hrs_sum', 'vpd_afternoon_mean_mean']
    print(f"\nMissing values in key variables:")
    for var in key_vars:
        if var in df.columns:
            n_missing = df[var].isnull().sum()
            pct_missing = 100 * n_missing / len(df)
            print(f"  {var}: {n_missing} ({pct_missing:.1f}%)")
    
    # Show corrected statistics for hour columns
    print(f"\n✓ Hour column statistics (after conversion):")
    for col in hour_columns:
        if col in df.columns:
            print(f"  {col}: mean={df[col].mean():.1f} hrs/week, range=[{df[col].min():.1f}, {df[col].max():.1f}]")
    
    # Verify data types
    print(f"\n✓ Data types verified:")
    for col in hour_columns + numeric_columns:
        if col in df.columns:
            print(f"  {col}: {df[col].dtype}")
    
    print(f"\nFirst few rows:")
    display(df.head())
    
else:
    print(f"⚠️  Data file not found: {data_file}")
    print(f"\nPlease run notebook 01b_experimental_design_from_processed.ipynb first")
    print(f"Or update the filename to match your processed data.")
    df = None

## 2. Descriptive Statistics

Summary statistics for key variables by region.

In [ ]:
if df is not None:
    # Define variable groups
    outcome_vars = ['slaughter_beef_dairy', 'log_slaughter_beef_dairy']
    climate_vars = [
        'day_above_30c_hrs_sum', 'day_above_35c_hrs_sum',
        'night_bad_hrs_sum', 'night_poor_recovery_hrs_sum',
        'vpd_afternoon_mean_mean', 'tas'
    ]
    
    # Overall summary
    print("=" * 80)
    print("OVERALL DESCRIPTIVE STATISTICS")
    print("=" * 80)
    display(df[outcome_vars + climate_vars].describe())
    
    # By region
    print("\n" + "=" * 80)
    print("STATISTICS BY REGION")
    print("=" * 80)
    
    for region in df['region'].unique():
        region_df = df[df['region'] == region]
        print(f"\n{region.upper().replace('_', ' ')}:")
        print("-" * 80)
        display(region_df[outcome_vars + climate_vars].describe())

## 3. Time Series Visualization

Plot key variables over time for both regions.

In [ ]:
if df is not None:
    fig, axes = plt.subplots(3, 2, figsize=(16, 12))
    fig.suptitle('Climate Variables and Cattle Slaughter Over Time', fontsize=16, fontweight='bold')
    
    # Define variables to plot
    plot_vars = [
        ('slaughter_beef_dairy', 'Cattle Slaughter (head)', False),
        ('log_slaughter_beef_dairy', 'Log Cattle Slaughter', False),
        ('day_above_30c_hrs_sum', 'Daytime Hours Above 30°C (weekly sum)', True),
        ('night_bad_hrs_sum', 'Nighttime Hours Above 24°C (weekly sum)', True),
        ('vpd_afternoon_mean_mean', 'VPD Mean (kPa)', True),
        ('tas', 'Thermal Acclimation State', True)
    ]
    
    for idx, (var, title, show_legend) in enumerate(plot_vars):
        ax = axes[idx // 2, idx % 2]
        
        for region in df['region'].unique():
            region_df = df[df['region'] == region].sort_values('week_ending')
            ax.plot(region_df['week_ending'], region_df[var], 
                   marker='o', label=region.replace('_', ' ').title(), linewidth=2)
        
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlabel('Week Ending', fontsize=10)
        ax.set_ylabel(var.replace('_', ' ').title(), fontsize=10)
        ax.grid(True, alpha=0.3)
        
        if show_legend:
            ax.legend(loc='best', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Time series plots generated")

## 4. Correlation Analysis

Examine correlations between climate variables and cattle slaughter.

In [ ]:
if df is not None:
    # Select variables for correlation
    corr_vars = [
        'log_slaughter_beef_dairy',
        'day_above_30c_hrs_sum',
        'day_above_35c_hrs_sum',
        'night_bad_hrs_sum',
        'night_poor_recovery_hrs_sum',
        'vpd_afternoon_mean_mean',
        'tas'
    ]
    
    # Overall correlation
    print("=" * 80)
    print("OVERALL CORRELATION MATRIX")
    print("=" * 80)
    corr_matrix = df[corr_vars].corr()
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', 
                center=0, vmin=-1, vmax=1, square=True, ax=ax,
                cbar_kws={'label': 'Correlation'})
    ax.set_title('Correlation Matrix: Climate Variables and Log Cattle Slaughter', 
                fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()
    
    # Correlation by region
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle('Correlation Matrices by Region', fontsize=16, fontweight='bold')
    
    for idx, region in enumerate(df['region'].unique()):
        region_df = df[df['region'] == region]
        corr_matrix_region = region_df[corr_vars].corr()
        
        sns.heatmap(corr_matrix_region, annot=True, fmt='.3f', cmap='coolwarm',
                   center=0, vmin=-1, vmax=1, square=True, ax=axes[idx],
                   cbar_kws={'label': 'Correlation'})
        axes[idx].set_title(f"{region.replace('_', ' ').title()}", 
                          fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Correlation analysis complete")

## 5. Scatter Plots: Heat Stress vs. Cattle Slaughter

Visualize relationships between key heat stress indicators and cattle slaughter.

In [ ]:
if df is not None:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Heat Stress Indicators vs. Log Cattle Slaughter', 
                fontsize=16, fontweight='bold')
    
    # Define predictors to plot
    predictors = [
        ('day_above_30c_hrs_sum', 'Daytime Hours >30°C (weekly)'),
        ('day_above_35c_hrs_sum', 'Daytime Hours >35°C (weekly)'),
        ('night_bad_hrs_sum', 'Night Hours >24°C (weekly)'),
        ('night_poor_recovery_hrs_sum', 'Night Hours >21°C (weekly)'),
        ('vpd_afternoon_mean_mean', 'VPD Mean (kPa)'),
        ('tas', 'Thermal Acclimation State')
    ]
    
    colors = {'region_4': '#1f77b4', 'region_6': '#ff7f0e'}
    
    for idx, (var, label) in enumerate(predictors):
        ax = axes[idx // 3, idx % 3]
        
        for region in df['region'].unique():
            region_df = df[df['region'] == region]
            ax.scatter(region_df[var], region_df['log_slaughter_beef_dairy'],
                      alpha=0.6, s=50, label=region.replace('_', ' ').title(),
                      color=colors.get(region, 'gray'))
            
            # Add regression line
            if len(region_df) > 1:
                z = np.polyfit(region_df[var].dropna(), 
                              region_df.loc[region_df[var].notna(), 'log_slaughter_beef_dairy'], 1)
                p = np.poly1d(z)
                x_line = np.linspace(region_df[var].min(), region_df[var].max(), 100)
                ax.plot(x_line, p(x_line), '--', color=colors.get(region, 'gray'), 
                       alpha=0.8, linewidth=2)
        
        ax.set_xlabel(label, fontsize=10)
        ax.set_ylabel('Log Cattle Slaughter', fontsize=10)
        ax.legend(loc='best', fontsize=8)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Scatter plots generated")

## 6. Simple Regression Models

Fit simple OLS models for each heat stress indicator.

In [ ]:
if df is not None and HAS_STATSMODELS:
    print("=" * 80)
    print("SIMPLE OLS REGRESSION RESULTS")
    print("=" * 80)
    print("Model: log_slaughter_beef_dairy ~ predictor + region")
    print("\n")
    
    # Define predictors
    predictors = [
        'day_above_30c_hrs_sum',
        'day_above_35c_hrs_sum',
        'night_bad_hrs_sum',
        'night_poor_recovery_hrs_sum',
        'vpd_afternoon_mean_mean',
        'tas'
    ]
    
    results_summary = []
    
    for predictor in predictors:
        # Drop missing values
        model_df = df[['log_slaughter_beef_dairy', predictor, 'region']].dropna().copy()
        
        if len(model_df) > 0:
            # Create dummy variable for region
            model_df = pd.get_dummies(model_df, columns=['region'], drop_first=True)
            
            # Ensure all columns are float64 (critical for statsmodels)
            for col in model_df.columns:
                if col != 'region':  # Skip if there's a region column left
                    model_df[col] = pd.to_numeric(model_df[col], errors='coerce').astype(np.float64)
            
            # Prepare variables
            y = model_df['log_slaughter_beef_dairy'].values.astype(np.float64)
            X_cols = [predictor] + [c for c in model_df.columns if c.startswith('region_')]
            X = model_df[X_cols].values.astype(np.float64)
            X = add_constant(X, has_constant='add')
            
            # Fit model
            model = OLS(y, X).fit()
            
            # Extract coefficient and stats (predictor is at index 1 after constant)
            coef = model.params[1]
            se = model.bse[1]
            t_stat = model.tvalues[1]
            p_value = model.pvalues[1]
            ci_lower, ci_upper = model.conf_int()[1]
            
            results_summary.append({
                'Predictor': predictor,
                'Coefficient': coef,
                'Std Error': se,
                't-statistic': t_stat,
                'p-value': p_value,
                '95% CI Lower': ci_lower,
                '95% CI Upper': ci_upper,
                'R-squared': model.rsquared,
                'N': int(model.nobs)
            })
            
            print(f"\n{predictor}:")
            print("-" * 80)
            print(f"  Coefficient: {coef:.6f}")
            print(f"  Std Error: {se:.6f}")
            print(f"  t-statistic: {t_stat:.3f}")
            print(f"  p-value: {p_value:.4f}")
            print(f"  95% CI: [{ci_lower:.6f}, {ci_upper:.6f}]")
            print(f"  R-squared: {model.rsquared:.4f}")
            print(f"  Observations: {int(model.nobs)}")
    
    # Create summary table
    results_df = pd.DataFrame(results_summary)
    
    print("\n" + "=" * 80)
    print("SUMMARY TABLE")
    print("=" * 80)
    display(results_df)
elif df is not None:
    print("⚠️  Skipping regression analysis (statsmodels not available)")
    print("   Please fix package versions and restart kernel.")

## 7. Panel Regression with Fixed Effects

Account for region-specific characteristics using fixed effects.

In [ ]:
if df is not None and HAS_STATSMODELS:
    print("=" * 80)
    print("PANEL REGRESSION WITH REGION FIXED EFFECTS")
    print("=" * 80)
    print("Model: log_slaughter ~ daytime_heat + nighttime_heat + VPD + TAS")
    print("\n")
    
    # Prepare data for panel regression
    panel_df = df[[
        'region', 'week_ending', 'log_slaughter_beef_dairy',
        'day_above_30c_hrs_sum', 'night_bad_hrs_sum', 
        'vpd_afternoon_mean_mean', 'tas'
    ]].dropna().copy()
    
    if len(panel_df) > 0:
        
        if HAS_PANEL_OLS:
            # Use linearmodels PanelOLS
            print("Using linearmodels.panel.PanelOLS\n")
            
            # Set multi-index for panel data
            panel_df = panel_df.set_index(['region', 'week_ending'])
            
            # Prepare variables
            y = panel_df['log_slaughter_beef_dairy']
            X = panel_df[[
                'day_above_30c_hrs_sum', 'night_bad_hrs_sum',
                'vpd_afternoon_mean_mean', 'tas'
            ]]
            
            # Fit fixed effects model
            fe_model = PanelOLS(y, X, entity_effects=True).fit()
            
            print(fe_model.summary)
            
            print("\n" + "=" * 80)
            print("MODEL INTERPRETATION")
            print("=" * 80)
            print("\nCoefficient Interpretation (holding other variables constant):")
            print("-" * 80)
            
            for var in X.columns:
                coef = fe_model.params[var]
                pval = fe_model.pvalues[var]
                sig = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
                
                # Convert to percentage change
                pct_change = (np.exp(coef) - 1) * 100
                
                print(f"\n{var}:")
                print(f"  Coefficient: {coef:.6f} {sig}")
                print(f"  Interpretation: A 1-unit increase in {var}")
                print(f"                 is associated with a {pct_change:.3f}% change in cattle slaughter")
        
        else:
            # Fallback: Use OLS with region dummy variables (fixed effects)
            print("Using OLS with region fixed effects (dummy variables)\n")
            
            # Create region dummies
            panel_df_fe = pd.get_dummies(panel_df, columns=['region'], drop_first=True)
            
            # Prepare variables
            y = panel_df_fe['log_slaughter_beef_dairy']
            X_vars = [
                'day_above_30c_hrs_sum', 'night_bad_hrs_sum',
                'vpd_afternoon_mean_mean', 'tas'
            ]
            region_dummies = [c for c in panel_df_fe.columns if c.startswith('region_')]
            
            X = panel_df_fe[X_vars + region_dummies]
            X = add_constant(X)
            
            # Fit OLS model
            fe_model = OLS(y, X).fit()
            
            print(fe_model.summary())
            
            print("\n" + "=" * 80)
            print("MODEL INTERPRETATION")
            print("=" * 80)
            print("\nCoefficient Interpretation (holding other variables constant):")
            print("-" * 80)
            
            for var in X_vars:
                coef = fe_model.params[var]
                pval = fe_model.pvalues[var]
                sig = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
                
                # Convert to percentage change
                pct_change = (np.exp(coef) - 1) * 100
                
                print(f"\n{var}:")
                print(f"  Coefficient: {coef:.6f} {sig}")
                print(f"  Interpretation: A 1-unit increase in {var}")
                print(f"                 is associated with a {pct_change:.3f}% change in cattle slaughter")
    else:
        print("⚠️  Insufficient data for panel regression")
elif df is not None:
    print("⚠️  Skipping panel regression (statsmodels not available)")
    print("   Please fix package versions and restart kernel.")

## 8. Lagged Effects Analysis

Examine delayed effects of heat stress on cattle slaughter.

In [ ]:
if df is not None and HAS_STATSMODELS:
    # Find lagged variables
    lag_vars = [c for c in df.columns if 'lag' in c and 'day_above_30c' in c]
    
    if len(lag_vars) > 0:
        print("=" * 80)
        print("LAGGED EFFECTS ANALYSIS")
        print("=" * 80)
        print(f"Analyzing lagged effects of daytime heat (>30°C)")
        print(f"Found {len(lag_vars)} lagged variables\n")
        
        # Fit models with different lags
        lag_results = []
        
        # Current effect
        base_df = df[['log_slaughter_beef_dairy', 'day_above_30c_hrs_sum', 'region']].dropna().copy()
        if len(base_df) > 0:
            base_df = pd.get_dummies(base_df, columns=['region'], drop_first=True)
            
            # Ensure numeric dtypes
            for col in base_df.columns:
                base_df[col] = pd.to_numeric(base_df[col], errors='coerce').astype(np.float64)
            
            y = base_df['log_slaughter_beef_dairy'].values.astype(np.float64)
            X = base_df.drop('log_slaughter_beef_dairy', axis=1).values.astype(np.float64)
            X = add_constant(X, has_constant='add')
            
            model = OLS(y, X).fit()
            
            lag_results.append({
                'Lag': 'Current',
                'Variable': 'day_above_30c_hrs_sum',
                'Coefficient': model.params[1],
                'Std Error': model.bse[1],
                'p-value': model.pvalues[1]
            })
        
        # Lagged effects
        for lag_var in sorted(lag_vars):
            lag_num = lag_var.split('_lag')[-1]
            
            model_df = df[['log_slaughter_beef_dairy', lag_var, 'region']].dropna().copy()
            if len(model_df) > 0:
                model_df = pd.get_dummies(model_df, columns=['region'], drop_first=True)
                
                # Ensure numeric dtypes
                for col in model_df.columns:
                    model_df[col] = pd.to_numeric(model_df[col], errors='coerce').astype(np.float64)
                
                y = model_df['log_slaughter_beef_dairy'].values.astype(np.float64)
                X = model_df.drop('log_slaughter_beef_dairy', axis=1).values.astype(np.float64)
                X = add_constant(X, has_constant='add')
                
                model = OLS(y, X).fit()
                
                lag_results.append({
                    'Lag': f'Lag {lag_num}',
                    'Variable': lag_var,
                    'Coefficient': model.params[1],
                    'Std Error': model.bse[1],
                    'p-value': model.pvalues[1]
                })
        
        # Create summary DataFrame
        lag_df = pd.DataFrame(lag_results)
        
        print("\nLagged Effects Summary:")
        display(lag_df)
        
        # Plot lagged effects
        fig, ax = plt.subplots(figsize=(10, 6))
        
        x_pos = range(len(lag_df))
        ax.errorbar(x_pos, lag_df['Coefficient'], 
                   yerr=1.96 * lag_df['Std Error'],
                   marker='o', markersize=8, capsize=5, capthick=2,
                   linewidth=2, label='95% CI')
        ax.axhline(y=0, color='red', linestyle='--', alpha=0.5, linewidth=1)
        
        ax.set_xticks(x_pos)
        ax.set_xticklabels(lag_df['Lag'])
        ax.set_xlabel('Lag Period', fontsize=12)
        ax.set_ylabel('Coefficient (log points)', fontsize=12)
        ax.set_title('Lagged Effects of Daytime Heat (>30°C) on Cattle Slaughter',
                    fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.legend()
        
        plt.tight_layout()
        plt.show()
        
        print("\n✓ Lagged effects analysis complete")
    else:
        print("⚠️  No lagged variables found in dataset")
elif df is not None:
    print("⚠️  Skipping lagged effects analysis (statsmodels not available)")
    print("   Please fix package versions and restart kernel.")

## 9. Regional Comparison

Compare heat stress effects between Region 4 (Southeast) and Region 6 (South Central).

In [ ]:
if df is not None and HAS_STATSMODELS:
    print("=" * 80)
    print("REGIONAL COMPARISON: REGION 4 VS REGION 6")
    print("=" * 80)
    print("\n")
    
    # Fit separate models for each region
    predictors = ['day_above_30c_hrs_sum', 'night_bad_hrs_sum', 
                  'vpd_afternoon_mean_mean', 'tas']
    
    regional_results = {}
    
    for region in df['region'].unique():
        region_df = df[df['region'] == region][['log_slaughter_beef_dairy'] + predictors].dropna().copy()
        
        if len(region_df) > 0:
            # Ensure all columns are numeric
            for col in region_df.columns:
                region_df[col] = pd.to_numeric(region_df[col], errors='coerce').astype(np.float64)
            
            y = region_df['log_slaughter_beef_dairy'].values.astype(np.float64)
            X = region_df[predictors].values.astype(np.float64)
            X = add_constant(X, has_constant='add')
            
            model = OLS(y, X).fit()
            
            # Store results with proper indexing
            params_dict = {predictors[i]: model.params[i+1] for i in range(len(predictors))}
            pvals_dict = {predictors[i]: model.pvalues[i+1] for i in range(len(predictors))}
            
            regional_results[region] = {
                'model': model,
                'coefs': params_dict,
                'pvals': pvals_dict,
                'rsquared': model.rsquared
            }
            
            print(f"{region.upper().replace('_', ' ')}:")
            print("-" * 80)
            print(model.summary())
            print("\n")
    
    # Compare coefficients
    if len(regional_results) == 2:
        print("\n" + "=" * 80)
        print("COEFFICIENT COMPARISON")
        print("=" * 80)
        
        comparison_data = []
        for var in predictors:
            row = {'Variable': var}
            for region, results in regional_results.items():
                row[f"{region}_coef"] = results['coefs'][var]
                row[f"{region}_pval"] = results['pvals'][var]
            comparison_data.append(row)
        
        comparison_df = pd.DataFrame(comparison_data)
        display(comparison_df)
        
        # Plot coefficient comparison
        fig, ax = plt.subplots(figsize=(12, 6))
        
        x = np.arange(len(predictors))
        width = 0.35
        
        regions = list(regional_results.keys())
        colors = ['#1f77b4', '#ff7f0e']
        
        for idx, region in enumerate(regions):
            coefs = [regional_results[region]['coefs'][var] for var in predictors]
            ax.bar(x + idx * width, coefs, width, 
                  label=region.replace('_', ' ').title(),
                  color=colors[idx], alpha=0.8)
        
        ax.set_xlabel('Predictor Variable', fontsize=12)
        ax.set_ylabel('Coefficient', fontsize=12)
        ax.set_title('Regional Comparison: Heat Stress Effects on Cattle Slaughter',
                    fontsize=14, fontweight='bold')
        ax.set_xticks(x + width / 2)
        ax.set_xticklabels([v.replace('_', ' ') for v in predictors], rotation=45, ha='right')
        ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        plt.show()
        
        print("\n✓ Regional comparison complete")
elif df is not None:
    print("⚠️  Skipping regional comparison (statsmodels not available)")
    print("   Please fix package versions and restart kernel.")

## 10. Summary and Key Findings

Synthesize main results from the analysis.

In [ ]:
if df is not None:
    print("=" * 80)
    print("ANALYSIS SUMMARY")
    print("=" * 80)
    
    print("\n📊 DATA COVERAGE:")
    print("-" * 80)
    print(f"  Period analyzed: {df['week_ending'].min().strftime('%Y-%m-%d')} to {df['week_ending'].max().strftime('%Y-%m-%d')}")
    print(f"  Total weeks: {len(df)}")
    print(f"  Regions: {', '.join([r.replace('_', ' ').title() for r in df['region'].unique()])}")
    
    print("\n🌡️  CLIMATE PATTERNS:")
    print("-" * 80)
    print(f"  Average daytime hours >30°C: {df['day_above_30c_hrs_sum'].mean():.1f} hrs/week")
    print(f"  Average nighttime hours >24°C: {df['night_bad_hrs_sum'].mean():.1f} hrs/week")
    print(f"  Average VPD: {df['vpd_afternoon_mean_mean'].mean():.2f} kPa")
    print(f"  Average TAS: {df['tas'].mean():.3f}")
    
    print("\n🐮 CATTLE SLAUGHTER:")
    print("-" * 80)
    print(f"  Average weekly slaughter: {df['slaughter_beef_dairy'].mean():,.0f} head")
    print(f"  Range: {df['slaughter_beef_dairy'].min():,.0f} - {df['slaughter_beef_dairy'].max():,.0f} head")
    
    print("\n🔬 STATISTICAL RELATIONSHIPS:")
    print("-" * 80)
    print("  Key correlations with log cattle slaughter:")
    for var in ['day_above_30c_hrs_sum', 'night_bad_hrs_sum', 'vpd_afternoon_mean_mean', 'tas']:
        corr = df[['log_slaughter_beef_dairy', var]].corr().iloc[0, 1]
        print(f"    {var}: {corr:.3f}")
    
    print("\n💡 NEXT STEPS:")
    print("-" * 80)
    print("  1. Extend analysis to full time period (1984-2025)")
    print("  2. Implement Distributed Lag Non-linear Models (DLNM) in R")
    print("  3. Explore non-linear effects and threshold responses")
    print("  4. Analyze seasonal patterns and long-term trends")
    print("  5. Include additional control variables (e.g., feed prices, market conditions)")
    print("  6. Investigate heterogeneous effects by farm size and management practices")
    
    print("\n" + "=" * 80)
    print("✓ ANALYSIS COMPLETE")
    print("=" * 80)